# pm4py-ucm — scenario synthesis tutorial

This tutorial teaches the **scenario synthesis** layer of `pm4py-ucm`: how to turn an event log into a `.jucm` file that jUCMNav can *step through* interactively, one scenario per behavioural variant discovered in the log.

It is the pedagogical companion to `scenario_synthesis.ipynb` (which is an end-to-end demonstration on a real log). Here we build up the concepts one at a time on the smallest examples that illustrate each idea, then apply them to a bundled real log at the end.

## What you will learn

1. Why classical sequence-variant analysis over-counts variants when the process has concurrency, and how the **choice signature** solves it.
2. How `discover_scenarios` populates the URN scenario layer — variables, per-loop integer counters, one `ScenarioDef` per variant.
3. Why loops need a **LoopEntryGuard** and how per-scenario iteration counts work.
4. The two condition-encoding strategies — **variant-driven** (lossless) and **data-driven** (business-readable) — and when each fits.
5. How decomposition is orthogonal to scenario synthesis (multi-map UCMs get the same scenario coverage as flat ones).
6. What the CSV reports contain and how to use them to audit the pipeline.

## Prerequisites

* `pm4py-ucm` installed (`pip install -e .` from the repo root) with the `pm4py` extra.
* `scikit-learn` for the data-driven strategy (`pip install scikit-learn`).
* Optional: the `graphviz` system binary for rendering — nothing in this tutorial requires it.

## Setup

In [1]:
import sys, shutil
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pm4py_ucm").is_dir():
    if (REPO_ROOT.parent / "pm4py_ucm").is_dir():
        REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import pm4py_ucm
from pm4py_ucm.algo.discovery.variants import clustering as _clustering
from pm4py_ucm.algo.discovery.scenarios import synthesis as _scenarios

OUT = Path("tutorial_output")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
print(f"pm4py-ucm loaded from {Path(pm4py_ucm.__file__).parent}")

pm4py-ucm loaded from C:\Users\jucmn\Claude\pm4py-ucm\pm4py_ucm


## §1 — The problem

URN's Use Case Maps (UCMs) support **executable scenarios**: named specifications that jUCMNav can walk through, resolving every OR-fork against a set of variable initialisations. Authoring these by hand for a real process is laborious — you need one scenario per meaningful behaviour, each with:

- variables initialised to the right values,
- entry/exit points wired to the right start/end nodes,
- fork conditions written so the traversal engine follows the intended path.

Meanwhile, process mining routinely extracts **behavioural variants** from event logs — equivalence classes of traces that share the same control flow. The natural question is: can we synthesize the URN scenario layer directly from those variants?

The answer is yes, but only after fixing three problems:

1. Naive sequence-equality variants over-count when the model has parallelism (see §2).
2. The URN loop construct wraps its body in a mandatory-first-iteration structure — we need a workaround for scenarios with 0 iterations (see §5).
3. Every OR-fork branch needs a *mutually exclusive and jointly exhaustive* condition, otherwise the traversal engine picks arbitrarily or gets stuck (see §6).

The rest of this notebook walks through those pieces in the order the pipeline uses them.

## §2 — Concurrency-aware variants

Consider a tiny process tree: `X → (Y ∥ Z) → (A × B) → W`. The `∥` is a parallel block, so `Y` and `Z` can interleave in either order.

Three observed sequences in the log:

| Sequence           | Frequency | Interpretation                            |
|--------------------|-----------|-------------------------------------------|
| `X-Y-Z-A-W`        | 60        | Y-then-Z, A branch                        |
| `X-Y-Z-B-W`        | 30        | Y-then-Z, B branch                        |
| `X-Z-Y-A-W`        | 10        | Z-then-Y, A branch — same behaviour as #1 |

Sequence-variant clustering would report **3 variants**. But sequences 1 and 3 are just different interleavings of the same parallel block — they represent the same *choice* (branch A, not B). Concurrency-aware clustering should report **2 variants**.

The insight: replay each trace on the discovered process tree, recording at every operator node which *choice* the trace made. For `∥` we sort the sub-signatures into a canonical order so interleaving order is erased. For `×` we record which branch. For loops we record a coarsened iteration count. The resulting tuple is the trace's **choice signature**. Two traces with equal signatures are in the same variant.

In [2]:
class T:
    """Duck-typed process tree node (matches pm4py's ProcessTree shape)."""
    def __init__(self, operator=None, label=None, children=None):
        self.operator = operator
        self.label = label
        self.children = children or []

tree = T(operator="->", children=[
    T(label="X"),
    T(operator="+", children=[T(label="Y"), T(label="Z")]),
    T(operator="X", children=[T(label="A"), T(label="B")]),
    T(label="W"),
])

log = (
    [(f"a{i}",     ["X", "Y", "Z", "A", "W"]) for i in range(60)]
  + [(f"b{i}",     ["X", "Y", "Z", "B", "W"]) for i in range(30)]
  + [(f"a_alt_{i}",["X", "Z", "Y", "A", "W"]) for i in range(10)]
)

result = _clustering.cluster(log, tree)
print(f"cases                     : {result.total_cases}")
print(f"sequence variants         : {result.sequence_variant_count}")
print(f"concurrency-aware variants: {len(result.variants)}")
print(f"compression ratio         : {result.compression_ratio:.3f}")
print(f"fitness                   : {result.fitness_percentage:.1%}")
print()
for v in result.variants:
    print(f"  {v.variant_id}  freq={v.frequency:>3}  {v.partial_order_expression}")

cases                     : 100
sequence variants         : 3
concurrency-aware variants: 2
compression ratio         : 0.667
fitness                   : 100.0%

  v1  freq= 70  X -> (Y || Z) -> [A] -> W
  v2  freq= 30  X -> (Y || Z) -> [B] -> W


**Checkpoint.** The output should show *3 sequence variants → 2 concurrency-aware variants*, compression 0.667. `v1` covers 70 cases (60 A-branch Y-first + 10 A-branch Z-first) and `v2` covers 30 cases (B-branch). Interleaving order was erased by the choice-signature canonicalisation.

### The `partial_order_expression` field

Each variant carries a compact textual rendering of the activated sub-tree. `X -> (Y || Z) -> [A] -> W` means "X, then Y and Z in either order, then choose branch A, then W". This is the string you'd read to a domain expert to describe the variant. `linearization_count` tells you how many sequential orderings this partial order admits — useful when comparing the concurrency-aware count against sequence-variant analysis.

## §3 — Your first scenario synthesis

`pm4py_ucm.discover_scenarios(log)` does everything end-to-end: discover the process tree, build the UCM, cluster the variants, populate the URN scenario layer. For our synthetic tree we already have both, so we skip the discovery step and call the underlying pieces directly:

In [3]:
ucm = pm4py_ucm.convert_to_ucm(tree)
group = _scenarios.synthesize_scenarios(ucm, tree, result)

print("enumeration types :", [(e.name, e.values) for e in ucm.enumeration_types])
print("variables         :", [(v.name, v.type) for v in ucm.variables])
print("scenario groups   :", [g.name for g in ucm.scenario_groups])
print()
for sc in group.scenarios:
    inits = ", ".join(f"{i.variable.name}={i.value}" for i in sc.initializations)
    print(f"  {sc.name}: {inits}")

enumeration types : [('VariantId', ['v1', 'v2'])]
variables         : [('variant_id', 'enumeration')]
scenario groups   : ['MinedScenarios']

  v1_A: variant_id=v1
  v2_B: variant_id=v2


The synthesizer created:

- one `EnumerationType` named `VariantId` with values `[v1, v2]`,
- one `Variable` named `variant_id` typed as that enum,
- one `ScenarioGroup` (default name `MinedScenarios`),
- one `ScenarioDef` per variant, each initialising `variant_id` to its own value.

The scenario names carry a short discriminator (`v1_A`, `v2_B`) derived from each variant's most distinguishing feature — makes the jUCMNav scenarios panel readable at a glance.

In [4]:
jucm_path = OUT / "tiny.jucm"
pm4py_ucm.write_ucm(ucm, str(jucm_path))

# Peek at the scenario layer in the exported XMI
text = jucm_path.read_text(encoding="utf-8")
for line in text.splitlines():
    if any(k in line for k in ("scenarioGroups", "variables", "enumerationTypes",
                                 "initializations", "variant_id ==")):
        print(line[:130])

    <scenarioGroups name="MinedScenarios" id="33">
        <initializations value="v1" variable="32"/>
        <initializations value="v2" variable="32"/>
    </scenarioGroups>
    <variables name="variant_id" id="32" type="enumeration" enumerationType="31"/>
    <enumerationTypes name="VariantId" id="31" values="v1,v2" instances="32"/>
      <nodes xsi:type="ucm.map:StartPoint" name="start" id="2" x="30" y="35" succ="//@urndef/@specDiagrams.0/@connections.0" scena
      <nodes xsi:type="ucm.map:EndPoint" name="end" id="3" x="1249" y="35" pred="//@urndef/@specDiagrams.0/@connections.1" scenari
        <condition label="branch0" expression="variant_id == v1"/>
        <condition label="branch1" expression="variant_id == v2"/>


The `<condition expression="variant_id == v1"/>` lines are exactly what jUCMNav needs to resolve the OR-fork's `A` vs `B` branch at runtime. Scenario `v1_A` initialises `variant_id = v1`, so the traversal engine picks the branch guarded by `variant_id == v1`.

## §4 — Anatomy of a scenario

Every `ScenarioDef` produced by the synthesizer contains three things:

1. **Initializations.** One `<initializations>` element per variable the scenario sets. For variant-driven, that's always `variant_id = <its id>` plus one `loop_counter_<n>` per loop node the variant traversed.
2. **Entry / exit references.** `ScenarioStartPoint` / `ScenarioEndPoint` pointing at the map's start / end nodes so jUCMNav knows where to begin and end. These are *mandatory-by-default* — if the traversal doesn't reach the end, jUCMNav flags it as an incomplete scenario.
3. **Description.** A human-readable summary: `Intent: …` (plain-English "covers N cases that take branch X, then Y, then Z"), followed by the partial-order expression, frequency, and a truncated case-id list.

Let's inspect the description of scenario `v1`:

In [5]:
sc = group.scenarios[0]
print(sc.description)

Intent: Covers 70 case(s) that takes the A branch.
Partial-order: X -> (Y || Z) -> [A] -> W
Frequency: 70 case(s); linearizations: 2; distinct sequences in log: 2.
Case IDs: a0, a1, a2, a3, a4, a5, a6, a7, a8, a9, ... (+60 more)


The `Intent:` line is what makes each scenario self-explanatory in the jUCMNav scenarios panel: a stakeholder reviewing the exported model sees "Quick Assessment → Approve → Claim Approved" rather than an opaque `v3`.

## §5 — Loops need a LoopEntryGuard

The URN loop construct wires `entry → LoopJoin → body → LoopFork → exit` with a redo arc from LoopFork back to LoopJoin. The body always runs at least once — you enter the LoopJoin, execute the body, and only then decide at the LoopFork whether to redo or exit.

That's fine for a scenario that runs the loop *at least once*, but breaks for a scenario that should run it *zero* times. If variant V never entered the loop in its traces, we want scenario V to bypass the loop entirely. Naive counter semantics can't express this: initialising `counter = 0` still runs the body once (which decrements to `-1`) and only then exits.

The synthesizer solves this by splicing a synthetic OR-fork — the **LoopEntryGuard** — between the loop's upstream arc and LoopJoin:

```
upstream ──► LoopEntryGuard ──[counter > 0]──► LoopJoin → body → LoopFork
                    │                                              │
                    │                                       redo   │
                    └──[counter <= 0]──► post-loop           ◄─────┘
```

The guard fires *only on initial entry*. Subsequent iterations re-enter via the LoopFork's redo arc directly into LoopJoin, skipping the guard. Result: `counter = k` produces exactly `k` body executions, including `k = 0`.

Let's try it on a loop:

In [6]:
loop_tree = T(operator="->", children=[
    T(label="Open"),
    T(operator="*", children=[T(label="Review"), T(label="Revise")]),
    T(label="Close"),
])

loop_log = (
    [(f"once_{i}",   ["Open", "Review", "Close"]) for i in range(20)]
  + [(f"twice_{i}",  ["Open", "Review", "Revise", "Review", "Close"]) for i in range(15)]
  + [(f"thrice_{i}", ["Open", "Review", "Revise", "Review", "Revise", "Review", "Close"]) for i in range(10)]
)

loop_ucm, loop_res = pm4py_ucm.discover_scenarios(
    loop_log, parameters={"process_tree": loop_tree},
    max_loop_iterations=2,  # cap counter init at 2 for tractable scenarios
)
print(f"variants: {len(loop_res.variants)}")
for v in loop_res.variants:
    print(f"  {v.variant_id}  freq={v.frequency:>3}  {v.partial_order_expression}")

print()
print("variables and initialisations:")
for sc in loop_ucm.scenario_groups[0].scenarios:
    inits = ", ".join(f"{i.variable.name}={i.value}" for i in sc.initializations)
    print(f"  {sc.name}: {inits}")

variants: 2
  v1  freq= 25  Open -> Review^>=2 -> Close
  v2  freq= 20  Open -> Review^1 -> Close

variables and initialisations:
  v1_TwoReview: variant_id=v1, Loop_Review=2
  v2_flow: variant_id=v2, Loop_Review=1


Notice:

- One `Loop_Review` **integer** variable was created (contextually named from the loop body).
- Coarsening groups the 15+10 multi-iteration cases into one `≥2` variant, so we get 2 variants: once, and "at least twice".
- Both variants initialise `Loop_Review` to the observed max, capped at `max_loop_iterations` (default 2). The `"twice"` variant initialises `Loop_Review = 2`, the `"once"` variant `Loop_Review = 1`. If a variant had never entered the loop it would initialise `Loop_Review = 0` and the LoopEntryGuard would bypass the body entirely.

**Why cap iterations?** A real trace might loop 30 times. Playing that scenario in jUCMNav step-by-step is painful and adds no insight beyond "loop fires more than once". `max_loop_iterations=2` keeps every scenario short while still exercising the `1 vs many` distinction. Pass `max_loop_iterations=None` to disable the cap.

## §6 — Two condition-encoding strategies

So far every OR-fork has been conditioned with `variant_id == V`. This is the **variant-driven** encoding — the default. It is *lossless*: replaying scenario V in jUCMNav exactly reproduces V's choice signature.

The downside is that `variant_id == v3 || variant_id == v7 || variant_id == v12` doesn't *explain* the branching decision. A stakeholder reading the model can't tell *why* those particular variants took the same branch.

The **data-driven** encoding trades losslessness for readability. For each outside-loop OR-fork, it trains a small `sklearn.DecisionTreeClassifier` on case-level attributes (claim amount, broker, country, …) predicting the branch each case took. The tree is then translated into a jUCMNav boolean expression:

```
Broker == Spot_Health_Insurance && Claim_Value <= 1417646 && Product_Group == Boat
```

Case-level attributes are lifted automatically from event-level columns that happen to be case-constant. Enumerations are one-hot encoded so the tree can split on any single value; real-number columns are scaled to integers.

Let's compare both on a small example with a perfect predictor:

In [7]:
import pandas as pd

# Same shape as §3 but XOR is now A/B and there's a case attribute
# "Category" that perfectly predicts the branch.
attr_tree = T(operator="->", children=[
    T(label="Start"),
    T(operator="X", children=[T(label="A"), T(label="B")]),
    T(label="End"),
])

rows = []
cases = (
    [(f"a{i}", "low",  ["Start", "A", "End"]) for i in range(20)]
  + [(f"b{i}", "high", ["Start", "B", "End"]) for i in range(15)]
)
for cid, cat, trace in cases:
    for j, act in enumerate(trace):
        rows.append({
            "case:concept:name": cid,
            "concept:name": act,
            "time:timestamp": pd.Timestamp("2026-01-01") + pd.Timedelta(j, "s"),
            "Category": cat,
        })
log_df = pd.DataFrame(rows)

# Variant-driven
ucm_v, res_v = pm4py_ucm.discover_scenarios(
    log_df, parameters={"process_tree": attr_tree},
    condition_strategy="variant",
)
# Data-driven
ucm_d, res_d = pm4py_ucm.discover_scenarios(
    log_df, parameters={"process_tree": attr_tree},
    condition_strategy="data-driven",
)

def or_fork_conditions(ucm):
    from pm4py_ucm.objects.ucm.obj import UCM
    out = []
    for m in ucm.maps:
        for n in m.nodes:
            if isinstance(n, UCM.OrFork) and n.name == "OrFork":
                for k, arc in enumerate(n.succ_connections):
                    cond = arc.condition
                    out.append((f"branch{k}", cond.expression if cond else "(true)"))
    return out

print("Variant-driven OR-fork conditions:")
for k, e in or_fork_conditions(ucm_v):
    print(f"  {k}: {e}")

print()
print("Data-driven OR-fork conditions:")
for k, e in or_fork_conditions(ucm_d):
    print(f"  {k}: {e}")

Variant-driven OR-fork conditions:
  branch0: variant_id == v1
  branch1: variant_id == v2

Data-driven OR-fork conditions:
  branch0: Category == low
  branch1: Category != low


Same behaviour, different conditions:

- **Variant-driven**: `variant_id == v1` on the A branch, `variant_id == v2` on the B branch. The synthesized `variant_id` variable drives everything.
- **Data-driven**: `Category == low` on the A branch, `Category == high` on the B branch. No `variant_id` variable at all — instead there's a `Category` enum variable, and each scenario initialises `Category` to the modal value for its variant.

**When data-driven abandons.** If the log carries no case-constant attributes that survive the type/cardinality filters, the data-driven path warns and leaves OR-fork conditions at the converter default `true`. There is *no* silent fallback to variant-driven — the two strategies are meant to be compared as alternatives.

**Inside-loop OR-forks in data-driven mode.** Case attributes are static per case, so they can't disambiguate *per-iteration* choices inside a loop body. The data-driven path therefore falls back, only for inside-loop XORs, to a deterministic `true` / `false` split (branch 0 → `true`, others → `false`). See the `condition_mining.csv` report — inside-loop forks are flagged with `skipped_reason=inside_loop`.

## §7 — Decomposition is orthogonal

pm4py-ucm can split a large process tree into a **root map plus plug-in maps** via static stubs (`decomposition="auto"` / `"aggressive"` / dict). Scenario synthesis works transparently on the multi-map result: OR-forks pushed into plug-in maps receive the same conditions they would in the flat case, and loops in plug-in maps get their LoopEntryGuard + decrement responsibility spliced into the plug-in.

The mechanism: every UCM `OrFork` / `OrJoin` / `LoopFork` / `LoopJoin` carries a stable id linking it back to the tree node it came from. The synthesizer looks up each fork by id rather than by position, so cross-map interleaving doesn't matter.

One extra detail: when a loop's post-loop continuation is a `Stub`, the LoopEntryGuard's bypass arc could otherwise land on the stub *unbound*. The synthesizer detects this case and splices an `OrJoin` before the stub so it keeps a single bound entry.

Let's check that flat and decomposed produce the same conditions:

In [8]:
# Reuse the loop_log from §5.
flat_ucm, _ = pm4py_ucm.discover_scenarios(
    loop_log, parameters={"process_tree": loop_tree},
    decomposition=None,
)
# Add a decomposable tail so decomposition actually splits something.
big_tree = T(operator="->", children=[
    T(operator="*", children=[T(label="Review"), T(label="Revise")]),
    T(operator="->", children=[
        T(label="Approve"), T(label="Notify"), T(label="Archive"), T(label="Close"),
    ]),
])
big_log = (
    [(f"once_{i}",  ["Review", "Approve", "Notify", "Archive", "Close"]) for i in range(15)]
  + [(f"twice_{i}", ["Review", "Revise", "Review", "Approve", "Notify", "Archive", "Close"]) for i in range(10)]
)
decomp_ucm, decomp_res = pm4py_ucm.discover_scenarios(
    big_log, parameters={"process_tree": big_tree},
    decomposition="auto",
)
print(f"decomposed UCM maps: {len(decomp_ucm.maps)} -> {[m.name for m in decomp_ucm.maps]}")
print(f"variants:            {len(decomp_res.variants)}")
print(f"scenarios:           {sum(len(g.scenarios) for g in decomp_ucm.scenario_groups)}")

# Confirm decrement lands on Review (not Approve, Notify, or any tail activity)
for r in decomp_ucm.responsibilities:
    if r.expression and "Loop_" in r.expression:
        print(f"decrement on responsibility: {r.name!r} -> {r.expression}")

decomposed UCM maps: 2 -> ['DiscoveredMap', 'Approve to Close']
variants:            2
scenarios:           2
decrement on responsibility: 'Review' -> Loop_Review = Loop_Review - 1;


The decrement lands on `Review` — the loop body activity — regardless of which map it ended up in after decomposition. That's the id-based correlation at work.

**Which decomposition to pick?** For scenarios, both `auto` and `aggressive` work; pick based on how you want the diagram to read (fewer / larger maps vs more / smaller maps). Flat (`decomposition=None`) is fine too if you don't need the visual split. See the top-level README's Hierarchical decomposition section for the full parameter shape.

## §8 — Reports and how to audit them

Three CSV reports ship alongside the `.jucm`:

1. **`variants.csv`** — one row per variant with frequency, sequence-variant count, linearization count, partial-order expression, and a truncated list of case IDs. Trailing rows for `noise` and `totals` (fitness + compression). This is the headline artefact for the empirical section of a paper.
2. **`case_variant_map.csv`** — one row per case, mapping case ID to variant ID (or `noise`). Joins cleanly against your own log to enrich downstream analysis.
3. **`condition_mining.csv`** — data-driven mode only. One row per `(OR-fork, branch)` pair carrying accuracy, sample size, feature set used, `skipped_reason` (e.g. `inside_loop`), and the post-minimisation expression emitted on the arc.

Let's produce all three for our loop example:

In [9]:
pm4py_ucm.write_ucm(loop_ucm, str(OUT / "loop.jucm"))
pm4py_ucm.write_variants_report(loop_res, str(OUT / "loop.variants.csv"))
pm4py_ucm.write_case_variant_map(loop_res, str(OUT / "loop.case_variant_map.csv"))

for p in sorted(OUT.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:>6,} bytes)")

print()
print("variants.csv:")
print(pd.read_csv(OUT / "loop.variants.csv").to_string(index=False))

  loop.case_variant_map.csv  (   519 bytes)
  loop.jucm  ( 7,303 bytes)
  loop.variants.csv  (   371 bytes)
  tiny.jucm  ( 9,072 bytes)

variants.csv:
variant_id  frequency  sequence_variants  linearization_count                                                  partial_order_expression                                       sample_case_ids
        v1         25                  2                  1.0                                               Open -> Review^>=2 -> Close twice_0, twice_1, twice_2, twice_3, twice_4, +20 more
        v2         20                  1                  1.0                                                 Open -> Review^1 -> Close      once_0, once_1, once_2, once_3, once_4, +15 more
    totals         45                  3                  NaN fitness=1.000, compression=0.667 (concurrency-variants/sequence-variants)                                                   NaN


**What to check when auditing a synthesis run:**

1. `fitness_percentage` — how many cases replayed cleanly. Anything below ~90% means your discovered tree is a poor fit for a chunk of the log; check the noise case IDs.
2. `compression_ratio` — how much concurrency-aware clustering bought you. Ratios well below 1.0 mean the log has substantial parallel behaviour that classical sequence-variant analysis was double-counting.
3. In data-driven mode, `condition_mining.csv`'s `accuracy` column tells you which forks are genuinely predictable from case attributes and which aren't. An accuracy near `1/n_branches` means the fork is essentially uncorrelated with the available attributes — an honest signal to surface to a domain expert.

## §9 — Where to go next

- **`scenario_synthesis.ipynb`** — the empirical companion notebook. Runs the full pipeline on the bundled `ClaimsPaymentLog` (78 K events, 24 variants) in both encodings side-by-side, with the paper's numbers.
- **`web/streamlit_app_v2.py`** — the Scenarios web UI. Upload an XES, tune the miner and decomposition, pick a strategy, download `.jucm` + all three reports. See `web/README.md`.
- **`pm4py_ucm.discover_scenarios`** — the one-call entry point. All of the pieces this tutorial called separately are wrapped in this function.
- **`pm4py_ucm.write_variants_report`, `write_case_variant_map`, `write_condition_mining_report`** — the three report writers.

In jUCMNav: open any of the `.jucm` files under `tutorial_output/`, right-click the URN spec, pick a scenario in the *Scenarios* panel, and hit *Run*. The traversal follows the conditions the synthesizer wrote.